# 🦜 BirdSense AI — Google Colab GPU Backend

**Run every cell top-to-bottom in order.** Each section is clearly labelled.

| Section | Purpose |
|---|---|
| §1 Setup | Mount Drive, verify GPU, install packages |
| §2 Dataset | Download training audio from Xeno-Canto |
| §3 Server | Start FastAPI backend + public tunnel |
| §4 Keepalive | Prevent Colab session timeout |

> ⚠️ **IMPORTANT:** Set `Runtime → Change runtime type → T4 GPU` before running anything.

---
## §1 — Setup
### Cell 1.1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

### Cell 1.2 — Verify GPU (must show Tesla T4)

In [ ]:
import subprocess, sys

# Check NVIDIA GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# Check PyTorch CUDA
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if cuda_ok else 'No GPU'
    print(f'PyTorch CUDA: {cuda_ok}')
    print(f'GPU name    : {gpu_name}')
    if not cuda_ok:
        print('\n❌ No GPU detected!')
        print('   Go to Runtime → Change runtime type → T4 GPU, then re-run all cells.')
    else:
        print(f'\n✅ GPU ready: {gpu_name}')
except ImportError:
    print('PyTorch not yet installed — will be installed in Cell 1.4')

### Cell 1.3 — Change directory to project folder

In [ ]:
import os

PROJECT_DIR = '/content/drive/MyDrive/Minor_Project_Colab'

if not os.path.exists(PROJECT_DIR):
    print(f'❌ Project directory not found: {PROJECT_DIR}')
    print('   Make sure Minor_Project-main is in the root of your Google Drive.')
else:
    os.chdir(PROJECT_DIR)
    print(f'✅ Working directory: {os.getcwd()}')
    print(f'   Files: {os.listdir(".")}')

### Cell 1.4 — Install ML packages (Colab GPU only)

This installs all heavy libraries. Takes ~3 minutes on first run, faster after that.

In [ ]:
import subprocess, sys

print('📦 Installing BirdSense AI dependencies...')
print('   This may take 3-5 minutes on first run.\n')

# Install from requirements_colab.txt
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements_colab.txt', '-q'],
    capture_output=True, text=True
)

if result.returncode == 0:
    print('✅ All packages installed successfully!')
else:
    print('⚠️  Some packages had warnings (usually fine):')
    print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)

# Quick verification
checks = [
    ('birdnetlib', 'BirdNET'),
    ('tensorflow', 'TensorFlow'),
    ('tensorflow_hub', 'TF-Hub'),
    ('librosa', 'librosa'),
    ('fastapi', 'FastAPI'),
    ('uvicorn', 'uvicorn'),
    ('noisereduce', 'noisereduce'),
]
print('\n🔍 Package check:')
for pkg, label in checks:
    try:
        __import__(pkg)
        print(f'   ✅ {label}')
    except ImportError:
        print(f'   ❌ {label} — MISSING')

---
## §2 — Dataset Collection (Xeno-Canto)

**Skip this section if you already have audio files in `input/`.**  
Run these cells only when you want to download fresh training data.

### Cell 2.1 — Configure API key and species list

In [ ]:
import os

# ── Xeno-Canto credentials ────────────────────────────────────────────────────
# SECURITY: Use Colab Secrets (left panel → 🔑 icon) instead of hardcoding.
# If using Colab Secrets, replace the line below with:
#   from google.colab import userdata
#   os.environ['XENOCANTO_API_KEY'] = userdata.get('XENOCANTO_API_KEY')
os.environ['XENOCANTO_API_KEY'] = '66a66df58dca4ebd5610998429c48ad70d89ee5b'

# ── Project paths ─────────────────────────────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/Minor_Project_Colab'
DATASET_DIR = os.path.join(BASE_DIR, 'input')
os.makedirs(DATASET_DIR, exist_ok=True)

# ── Target species (15 Indian + common species) ───────────────────────────────
TARGET_SPECIES = [
    # Common Indian species
    'Corvus splendens',       # House Crow
    'Acridotheres tristis',   # Common Myna
    'Passer domesticus',      # House Sparrow
    'Columba livia',          # Rock Pigeon
    'Psittacula krameri',     # Rose-ringed Parakeet
    'Pycnonotus cafer',       # Red-vented Bulbul
    'Eudynamys scolopaceus',  # Asian Koel
    'Copsychus saularis',     # Oriental Magpie-Robin
    'Halcyon smyrnensis',     # White-throated Kingfisher
    'Alcedo atthis',          # Common Kingfisher
    'Pavo cristatus',         # Indian Peafowl
    'Dicrurus macrocercus',   # Black Drongo
    'Streptopelia decaocto',  # Eurasian Collared Dove
    # Additional well-recorded species
    'Turdus merula',          # Common Blackbird
    'Parus major',            # Great Tit
]

# ── Download settings ─────────────────────────────────────────────────────────
MIN_DURATION        = 2    # seconds
MAX_DURATION        = 10   # seconds
ALLOWED_QUALITY     = ['A', 'B']
MAX_FILES_PER_SPECIES = 50

print(f'✅ Configured {len(TARGET_SPECIES)} target species')
print(f'   Dataset directory: {DATASET_DIR}')
print(f'   Max files per species: {MAX_FILES_PER_SPECIES}')

### Cell 2.2 — Define Xeno-Canto fetch helpers

In [ ]:
import requests, os, time
from tqdm import tqdm

API_KEY = os.environ.get('XENOCANTO_API_KEY')

def fetch_xenocanto_page(species_name: str, page: int):
    """Fetch one page of Xeno-Canto recordings for a species."""
    parts = species_name.split()
    if len(parts) < 2:
        return None
    genus, sp = parts[0], parts[1]
    try:
        resp = requests.get(
            'https://xeno-canto.org/api/3/recordings',
            params={'query': f'gen:{genus} sp:{sp}', 'key': API_KEY,
                    'per_page': 100, 'page': page},
            timeout=30
        )
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f'   ⚠️  Page {page} fetch error for {species_name}: {e}')
        return None


def is_valid_recording(record: dict) -> bool:
    """Check duration and quality filters."""
    try:
        mins, secs = record['length'].split(':')
        duration = int(mins) * 60 + int(secs)
        return MIN_DURATION <= duration <= MAX_DURATION and record['q'] in ALLOWED_QUALITY
    except Exception:
        return False


def download_species_audio(species_name: str):
    """Download up to MAX_FILES_PER_SPECIES quality-filtered recordings."""
    folder = species_name.replace(' ', '_')
    save_path = os.path.join(DATASET_DIR, folder)
    os.makedirs(save_path, exist_ok=True)

    downloaded = 0
    page = 1

    while downloaded < MAX_FILES_PER_SPECIES:
        data = fetch_xenocanto_page(species_name, page)
        if not data:
            break

        records = data.get('recordings', [])
        if not records:
            print(f'   No recordings on page {page} for {species_name}')
            break

        for record in records:
            if downloaded >= MAX_FILES_PER_SPECIES:
                break
            if not is_valid_recording(record):
                continue

            file_url = record.get('file', '')
            if not isinstance(file_url, str) or not file_url:
                continue
            if file_url.startswith('//'):
                file_url = 'https:' + file_url
            elif not file_url.startswith('http'):
                continue

            fname = f"XC{record['id']}.mp3"
            fpath = os.path.join(save_path, fname)
            if os.path.exists(fpath):
                downloaded += 1  # count already downloaded
                continue

            try:
                r = requests.get(file_url, stream=True, timeout=(10, 30))
                r.raise_for_status()
                with open(fpath, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=64 * 1024):
                        if chunk:
                            f.write(chunk)
                downloaded += 1
            except Exception as e:
                print(f'   ⚠️  Download failed {file_url}: {e}')

        num_pages = int(data.get('numPages', page))
        if page >= num_pages:
            break
        page += 1

    status = '✅' if downloaded > 0 else '⚠️ '
    print(f'   {status} {species_name}: {downloaded} files')
    return downloaded


# Quick API connectivity test
print('🔍 Testing Xeno-Canto API...')
test = fetch_xenocanto_page('Corvus splendens', 1)
if test:
    print(f'   ✅ API working — found {test["numRecordings"]} recordings for Corvus splendens')
else:
    print('   ❌ API test failed — check your API key')

### Cell 2.3 — Download all species audio

> ⏱️ This can take **10–30 minutes** depending on the number of species and network speed.  
> Already-downloaded files are skipped automatically — safe to re-run.

In [ ]:
import time

print(f'📥 Starting download for {len(TARGET_SPECIES)} species...\n')
t_start = time.time()

summary = {}
for i, species in enumerate(TARGET_SPECIES, 1):
    print(f'[{i:>2}/{len(TARGET_SPECIES)}] {species}')
    count = download_species_audio(species)
    summary[species] = count

elapsed = time.time() - t_start
total = sum(summary.values())

print(f'\n{"="*50}')
print(f'✅ Download complete in {elapsed/60:.1f} min')
print(f'   Total files: {total}')
print(f'   Species with 0 files: {[sp for sp, c in summary.items() if c == 0]}')

---
## §3 — Backend Server

**Run these cells every Colab session to start the API server.**

### Cell 3.1 — Pre-load all ML models into GPU memory

In [ ]:
import os, sys, time

os.chdir('/content/drive/MyDrive/Minor_Project_Colab')
sys.path.insert(0, os.getcwd())

print('🧠 Pre-warming all models into T4 GPU memory...')
print('   (This takes 2-4 minutes on first run of the session)\n')

t0 = time.time()

import eval_prebuilt
eval_prebuilt.load_models_once()

elapsed = time.time() - t0

print(f'\n✅ All models loaded in {elapsed:.1f}s')
print(f'   BirdNET : {"✅ Ready" if eval_prebuilt._cached_analyzer is not None else "❌ Failed"}')
print(f'   YAMNet  : {"✅ Ready" if eval_prebuilt._cached_yamnet_model is not None else "❌ Failed"}')
print(f'   Perch   : {"✅ Ready" if eval_prebuilt._cached_perch_model is not None else "❌ Failed"}')

### Cell 3.2 — Build the Next.js frontend (run once per session)

This builds the website so the FastAPI server can serve it at `/`.

In [ ]:
import subprocess, os, shutil

FRONTEND_DRIVE_DIR = '/content/drive/MyDrive/Minor_Project_Colab/frontend'
LOCAL_TEMP_DIR = '/content/temp_frontend'

if not os.path.exists(FRONTEND_DRIVE_DIR):
    print('❌ frontend/ directory not found.')
    print('   Make sure the project was fully synced to your Google Drive.')
else:
    if os.path.exists(LOCAL_TEMP_DIR): shutil.rmtree(LOCAL_TEMP_DIR)
    os.makedirs(LOCAL_TEMP_DIR, exist_ok=True)
    
    print('🚀 Copying frontend source to Colab local memory (fast SSD)...')
    for item in os.listdir(FRONTEND_DRIVE_DIR):
        if item in ['.next', 'node_modules', 'out']: continue
        s_path = os.path.join(FRONTEND_DRIVE_DIR, item)
        d_path = os.path.join(LOCAL_TEMP_DIR, item)
        if os.path.isdir(s_path):
            shutil.copytree(s_path, d_path)
        else:
            shutil.copy2(s_path, d_path)

    print('📦 Installing frontend dependencies locally (takes ~30s)...')
    r1 = subprocess.run(['npm', 'install', '--silent'], cwd=LOCAL_TEMP_DIR,
                        capture_output=True, text=True)
    if r1.returncode != 0:
        print(f'⚠️  npm install warnings: {r1.stderr[-500:]}')

    print('🔨 Building Next.js website locally (takes ~30s)...')
    r2 = subprocess.run(['npm', 'run', 'build'], cwd=LOCAL_TEMP_DIR,
                        capture_output=True, text=True)

    if r2.returncode == 0:
        print('✅ Frontend built successfully!')
        local_out = os.path.join(LOCAL_TEMP_DIR, 'out')
        drive_out = os.path.join(FRONTEND_DRIVE_DIR, 'out')
        
        print('📤 Copying built files back to Google Drive...')
        if os.path.exists(drive_out): shutil.rmtree(drive_out)
        shutil.copytree(local_out, drive_out)
        print(f'✅ Website successfully exported to Drive: {drive_out}')
        shutil.rmtree(LOCAL_TEMP_DIR, ignore_errors=True)
    else:
        print('❌ Build failed:')
        print(r2.stderr[-1000:])

### Cell 3.3 — Start the FastAPI backend server

In [ ]:
import subprocess, time, requests as req, os

os.chdir('/content/drive/MyDrive/Minor_Project_Colab')

# Kill any existing uvicorn instance
subprocess.run(['pkill', '-f', 'uvicorn'], capture_output=True)
time.sleep(1)

print('🚀 Starting FastAPI server on port 8000...')
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'colab_api:app',
     '--host', '0.0.0.0', '--port', '8000',
     '--log-level', 'warning'],
    cwd='/content/drive/MyDrive/Minor_Project_Colab',
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for server to come up
for attempt in range(12):
    time.sleep(2)
    try:
        resp = req.get('http://localhost:8000/health', timeout=3)
        if resp.status_code == 200:
            health = resp.json()
            print(f'✅ Server is UP!')
            print(f'   GPU  : {health.get("gpu_devices", ["unknown"])[0]}')
            models = health.get('models_loaded', {})
            print(f'   BirdNET : {"✅" if models.get("BirdNET") else "⏳"}')
            print(f'   YAMNet  : {"✅" if models.get("YAMNet")  else "⏳"}')
            print(f'   Perch   : {"✅" if models.get("Perch")   else "⏳"}')
            break
    except Exception:
        print(f'   Waiting... ({attempt + 1}/12)')
else:
    print('❌ Server did not start. Check logs above.')
    # Print any startup errors
    try:
        stderr_out = server_proc.stderr.read(2000).decode()
        if stderr_out:
            print('Server stderr:', stderr_out)
    except Exception:
        pass

### Cell 3.4 — Start public tunnel & display URL

This cell tries **ngrok** first (most reliable), then falls back to **localtunnel**.  
The public URL is printed in a large, easy-to-copy box at the end.

In [ ]:
import subprocess, time, os, re, threading

PUBLIC_URL = None

# ── Method 1: Try pyngrok (pip install pyngrok, works without account for basic tunnels) ──
try:
    from pyngrok import ngrok, conf
    # If you have an ngrok auth token, add it to Colab Secrets as 'NGROK_TOKEN'
    try:
        from google.colab import userdata
        token = userdata.get('NGROK_TOKEN')
        if token:
            ngrok.set_auth_token(token)
            print('🔑 ngrok auth token set from Colab Secrets')
    except Exception:
        pass

    tunnel = ngrok.connect(8000, 'http')
    PUBLIC_URL = tunnel.public_url
    print(f'✅ ngrok tunnel active')

except Exception as e:
    print(f'⚠️  ngrok not available ({e}), trying localtunnel...')

    # ── Method 2: localtunnel ──
    try:
        subprocess.run(['npm', 'install', '-g', 'localtunnel', '--silent'],
                       capture_output=True)

        lt_proc = subprocess.Popen(
            ['lt', '--port', '8000'],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )

        # Read URL from stdout
        for _ in range(10):
            line = lt_proc.stdout.readline()
            if 'loca.lt' in line or 'ngrok' in line:
                match = re.search(r'https?://[^\s]+', line)
                if match:
                    PUBLIC_URL = match.group(0)
                    print(f'✅ localtunnel active')
                    break
            time.sleep(1)

        if not PUBLIC_URL:
            print('⚠️  localtunnel started but URL not detected. Try running it manually:')
            print('   !lt --port 8000')
    except Exception as e2:
        print(f'❌ localtunnel also failed: {e2}')


# ── Display URL in a prominent box ──────────────────────────────────────────
if PUBLIC_URL:
    from IPython.display import display, HTML
    display(HTML(f"""
    <div style="
        background: linear-gradient(135deg, #0a0e1a, #1e1b4b);
        border: 2px solid #CCFF00;
        border-radius: 16px;
        padding: 28px 32px;
        margin: 16px 0;
        font-family: monospace;
        box-shadow: 0 0 30px rgba(204,255,0,0.25);
    ">
        <p style="color: #94a3b8; font-size: 13px; margin: 0 0 8px 0;">🌐 BirdSense AI Public URL</p>
        <p style="color: #CCFF00; font-size: 22px; font-weight: bold; margin: 0 0 12px 0; word-break: break-all;">
            {PUBLIC_URL}
        </p>
        <p style="color: #64748b; font-size: 12px; margin: 0;">
            📋 Copy this URL → paste into the website's <b style='color:#a5b4fc'>Colab GPU Connection</b> field<br>
            🌐 Or open this URL directly in your browser to access the full website
        </p>
    </div>
    """))
    print(f'\nPUBLIC URL: {PUBLIC_URL}')
else:
    print('\n⚠️  No public URL generated.')
    print('   Run this command in a new Colab cell to start a tunnel manually:')
    print('   !npx localtunnel --port 8000')

### Cell 3.5 — Health check (verify everything is ready)

In [ ]:
import requests as req
from IPython.display import display, HTML

try:
    resp = req.get('http://localhost:8000/health', timeout=5)
    h = resp.json()

    models = h.get('models_loaded', {})
    gpus   = h.get('gpu_devices', ['unknown'])

    def badge(ok): return '✅ Loaded' if ok else '⏳ Loading'

    display(HTML(f"""
    <div style="background:#050814; border:1px solid #1e293b; border-radius:12px;
                padding:20px 24px; font-family:monospace; color:#f1f5f9;">
        <h3 style="color:#CCFF00; margin:0 0 16px 0;">🔬 BirdSense AI — System Status</h3>
        <table style="width:100%; border-collapse:collapse;">
            <tr><td style="color:#64748b; padding:4px 0;">Server</td>
                <td style="color:#4ade80; font-weight:bold;">✅ Running on port 8000</td></tr>
            <tr><td style="color:#64748b; padding:4px 0;">GPU</td>
                <td style="color:#a5b4fc;">{', '.join(gpus)}</td></tr>
            <tr><td style="color:#64748b; padding:4px 0;">BirdNET (Lead, 60%)</td>
                <td style="color:{'#4ade80' if models.get('BirdNET') else '#fbbf24'}">{badge(models.get('BirdNET'))}</td></tr>
            <tr><td style="color:#64748b; padding:4px 0;">YAMNet (Validator, 25%)</td>
                <td style="color:{'#4ade80' if models.get('YAMNet') else '#fbbf24'}">{badge(models.get('YAMNet'))}</td></tr>
            <tr><td style="color:#64748b; padding:4px 0;">Perch (Reviewer, 15%)</td>
                <td style="color:{'#4ade80' if models.get('Perch') else '#fbbf24'}">{badge(models.get('Perch'))}</td></tr>
            <tr><td style="color:#64748b; padding:4px 0;">Timestamp</td>
                <td style="color:#64748b;">{h.get('timestamp','')}</td></tr>
        </table>
    </div>
    """))

    all_loaded = all(models.values())
    if all_loaded:
        print('\n🟢 ALL SYSTEMS GO — Ready to analyze audio!')
    else:
        print('\n🟡 Some models still loading — wait 30 seconds and re-run this cell.')

except Exception as e:
    print(f'❌ Health check failed: {e}')
    print('   Make sure Cell 3.3 ran successfully and the server is up.')

---
## §4 — Session Keepalive

### Cell 4.1 — Prevent Colab timeout (runs every 25 minutes)

Google Colab disconnects sessions that appear idle. This cell pings the server every 25 minutes  
to keep it alive. **Run this cell once and leave it running in the background.**

In [ ]:
import time, threading, requests as req, datetime

PING_INTERVAL = 25 * 60   # 25 minutes
_keepalive_running = True

def keepalive_worker():
    """Ping the local health endpoint every 25 minutes."""
    session_pings = 0
    while _keepalive_running:
        time.sleep(PING_INTERVAL)
        if not _keepalive_running:
            break
        try:
            resp = req.get('http://localhost:8000/health', timeout=5)
            session_pings += 1
            ts = datetime.datetime.now().strftime('%H:%M:%S')
            status = '✅' if resp.status_code == 200 else '⚠️'
            print(f'[{ts}] {status} Keepalive ping #{session_pings} — server alive')
        except Exception as e:
            ts = datetime.datetime.now().strftime('%H:%M:%S')
            print(f'[{ts}] ⚠️  Keepalive ping failed: {e}')


# Start in a daemon thread so it doesn't block the notebook
keepalive_thread = threading.Thread(target=keepalive_worker, daemon=True)
keepalive_thread.start()

print(f'✅ Keepalive started — pinging every {PING_INTERVAL//60} minutes')
print(f'   This keeps your Colab session alive during long analysis jobs.')
print(f'   The thread runs in the background — you can run other cells normally.')

---
## §5 — Optional Utilities

### Cell 5.1 — Quick GPU memory status

In [ ]:
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.used,memory.free,memory.total,utilization.gpu',
     '--format=csv,noheader,nounits'],
    capture_output=True, text=True
)
if result.stdout:
    name, used, free, total, util = result.stdout.strip().split(', ')
    print(f'GPU      : {name}')
    print(f'Memory   : {used} MB used / {total} MB total ({free} MB free)')
    print(f'Util     : {util}%')
    pct = int(used) / int(total) * 100
    bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
    print(f'Usage    : [{bar}] {pct:.1f}%')
else:
    print('nvidia-smi not available')

### Cell 5.2 — Restart server (if it crashes)

In [ ]:
import subprocess, time, requests as req, os

print('🔄 Restarting FastAPI server...')
subprocess.run(['pkill', '-f', 'uvicorn'], capture_output=True)
time.sleep(2)

os.chdir('/content/drive/MyDrive/Minor_Project_Colab')

server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'colab_api:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'warning'],
    cwd='/content/drive/MyDrive/Minor_Project_Colab',
)

for i in range(12):
    time.sleep(2)
    try:
        resp = req.get('http://localhost:8000/health', timeout=3)
        if resp.status_code == 200:
            print('✅ Server restarted successfully!')
            break
    except Exception:
        print(f'   Waiting... ({i+1}/12)')
else:
    print('❌ Server failed to restart')

### Cell 5.3 — Test with sample audio (sanity check)

In [ ]:
import requests as req, json, os

SAMPLE = '/content/drive/MyDrive/Minor_Project_Colab/sample_bird_chirp.wav'

if not os.path.exists(SAMPLE):
    print(f'❌ Sample file not found: {SAMPLE}')
else:
    print(f'📤 Sending sample audio to /analyze...')
    with open(SAMPLE, 'rb') as f:
        resp = req.post(
            'http://localhost:8000/analyze',
            files={'file': ('sample_bird_chirp.wav', f, 'audio/wav')},
            data={'n_sources': 2, 'week': -1},
            timeout=120
        )

    if resp.status_code == 200:
        data = resp.json()
        run_id = data.get('run_id')
        print(f'✅ Job submitted — run_id: {run_id}')
        print(f'   Stream progress at: http://localhost:8000/stream/{run_id}')

        # Poll for result
        import time
        for _ in range(30):
            time.sleep(5)
            r = req.get(f'http://localhost:8000/results/{run_id}', timeout=5)
            if r.status_code == 200:
                result = r.json()
                preds = result.get('predictions', [])
                print(f'\n🏆 Results ({result.get("elapsed_seconds", "?"):.1f}s):')
                for i, p in enumerate(preds[:5], 1):
                    print(f'   [{i}] {p.get("common_name", p.get("species"))} '
                          f'({p.get("species")}) — {p.get("confidence", 0)*100:.1f}%')
                break
            elif r.status_code == 202:
                print('   Still processing...')
        else:
            print('⏱️  Timed out waiting for results')
    else:
        print(f'❌ Error: HTTP {resp.status_code}')
        print(resp.text[:500])